# SNR Permuted MNIST: author-code reference harness

This user-supplied notebook invokes the authors' JAX implementation for reference verification. It is not the team's PyTorch reimplementation or evidence of completed team experiments.

## Reference and attribution

- Paper: [Self-Normalized Resets for Plasticity in Continual Learning, ICLR 2025](https://proceedings.iclr.cc/paper_files/paper/2025/file/359c0c4dc53b6bf64041b4c8334f1c11-Paper-Conference.pdf), Vivek Farias and Adam Jozefiak.
- Author repository: [ajozefiak/SelfNormalizedResets](https://github.com/ajozefiak/SelfNormalizedResets).
- The author README links a [reference Colab example](https://colab.research.google.com/drive/18jHVgPTH4CM9hlvnMvapbd4CMgi1x_2Q?usp=share_link). Exact export origin and author commit for this supplied notebook have not been verified.
- Respect the authors' repository license and notices when using their code. Core method implementation for the team must be written independently from the paper description.

## Status and execution boundary

The existing cells and saved outputs are retained as supplied. They were inspected statically, not executed or numerically validated in this update. Saved accuracy values and logs are unverified reference material, not team measurements; do not cite them as reproduced results in the deck/poster.

The cells clone external code, install its requirements, download MNIST through the author's loader, and configure a long sequential experiment. Execute reference work only in a separate Colab/scratch workspace or an external reference checkout with its own environment, never inside the team repository. Reference acquisition does not establish a course-compliant Hugging Face/Kaggle pipeline for the team.

Before any authorized reference execution, record the actual author commit, compatible dependencies, data provenance, agreed comparison settings, and later verified outputs/run links. The notebook's existing seed and settings are reference examples; official team configs, seed lists, and result conventions remain deferred.

See [author comparison plan](../docs/author_comparison.md), [setup plan](../docs/setup.md), and [data inventory](../docs/data_sources.md).


In [ ]:
!git clone https://github.com/ajozefiak/SelfNormalizedResets.git

Cloning into 'SelfNormalizedResets'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 30 (delta 0), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 18.20 KiB | 18.20 MiB/s, done.


In [ ]:
%cd SelfNormalizedResets

/content/SelfNormalizedResets


In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import SelfNormalizedResets
import jax
import jax.numpy as jnp
import jax.random as jr
import optax

# Experiment Config

In [ ]:
# Experiment Parameters
num_tasks = 2400
epochs = 1
batch_size = 16
num_classes = 10
N = 10000

# Logging
# log results every 100 tasks
save_freq = 100
verbose = True

# Seed
seed = 0
data_seed = seed
task_seed = seed
param_seed = seed

# Model
width = 100
# Choose optimzier_ from ["sgd", "adam"]
optimizer_ = "adam"
if optimizer_ == "sgd":
    lr = 1e-2
    optimizer = optax.sgd
elif optimizer_ == "adam":
    lr = 1e-3
    optimizer = optax.adam
reg_str = 0.0
# NOTE: Use alg = 3 for SNR and No-Intervention; use of integers over string due to jax.jit
alg = 3
# Train in parallel on a single GPU using vectorization (jax.vmap): [No-Intervention, SNR]
# Set a large reset-threshold for SNR before the first threshold-update.
tau_max = 20000
# Initial threshold: No-Intervention: threshold = infinity, SNR = tau_max
threshold = jnp.array([jnp.inf] + [tau_max])
# Threshold-update frequency: No-Intervention = update never, SNR = update every 16 tasks
threshold_reset_freq = jnp.array([jnp.inf, 16])
# threshold_percentile = 1 - eta (where eta is as defined in our paper): No-Intervention = 0.92 (could be any value), SNR = 0.92
threshold_percentile = jnp.array([0.92, 0.92])
# If the reset-threshold is too small given the choice of threshold_percentile and recent neuron-activation patterns, increase the threshold by a factor of threshold_expansion_factor
threshold_expansion_factor = 2
# Artificats from a more general implementation, remain in place due to use of jax.jit
reset_freq = 0
decay_rate = 0.0



# Run Experiment

In [ ]:
from SelfNormalizedResets import run_snr_permuted_mnist_experiment, MLP, init_fn_sgd, init_fn_adam, load_mnist, sample_subset_mnist

In [ ]:
# Load MNIST
data_key = jr.PRNGKey(data_seed)
(X_train, y_train), (X_test, y_test) = SelfNormalizedResets.load_mnist()
X_train_, y_train_ = sample_subset_mnist(X_train, y_train, N, data_key)

model = MLP(num_classes = num_classes, width = width)

threshold_file_name = f"Permuted_MNIST_SNR_{optimizer_}_seed_{seed}_lr_{lr}_thresholds"
file_name =  f"Permuted_MNIST_SNR_{optimizer_}_seed_{seed}_lr_{lr}.pkl"

if optimizer_ == "sgd":
    init_fn = init_fn_sgd
elif optimizer_ == "adam":
    init_fn = init_fn_adam

states = jax.vmap(init_fn, in_axes=(None, None, None, None, 0, None, None, None, None, 0, 0, None))(model,
                X_train[0].shape, param_seed, lr, threshold, reg_str, alg, reset_freq, decay_rate, threshold_reset_freq, threshold_percentile, threshold_expansion_factor)

run_snr_permuted_mnist_experiment(num_tasks, epochs, batch_size, seed, task_seed,
                            states, optimizer, model,
                            X_train_, y_train_, X_test, y_test, num_classes,
                            file_name, threshold_file_name, save_freq, verbose, tau_max)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Average Accuracy for Task 1: [0.8605 0.8605]
Average Accuracy for all tasks up to Task 1: [0.8605 0.8605]
Average Accuracy for Task 2: [0.8771 0.8771]
Average Accuracy for all tasks up to Task 2: [0.8688 0.8688]
Average Accuracy for Task 3: [0.8756 0.8765]
Average Accuracy for all tasks up to Task 3: [0.87106667 0.87136667]
Average Accuracy for Task 4: [0.8712 0.8728]
Average Accuracy for all tasks up to Task 4: [0.8711   0.871725]
Average Accuracy for Task 5: [0.8799 0.8795]
Average Accuracy for all tasks up to Task 5: [0.87286 0.87328]
Average Accuracy for Task 6: [0.8717 0.8767]
Average Accuracy for all tasks up to Task 6: [0.87266667 0.87385   ]
Average Accuracy for Task 7: [0.8692 0.8728]
Average Accuracy for all tasks up to Task 7: [0.87217143 0.8737    ]
Average Accuracy for Task 8: [0.8618 0.8643]
Average Accuracy for all tasks up to Task 8: [0.870875 0.872525]
Average Accuracy for Task 9: [0.8608 0.8743]
Average Accuracy for a